# Classification ML sur Embeddings LLM (DistilBERT)

**SAE-117** — 4 algorithmes ML sur embeddings DistilBERT pré-calculés (768 dimensions).

**Corrections vs version précédente :**
- Split 80/10/10 (au lieu de 80/20)
- Métriques en `macro` averaging (au lieu de `weighted`)
- GridSearchCV pour tous les modèles
- GaussianNB (embeddings continus, pas de MultinomialNB)

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

from src.constants import RANDOM_STATE, POLARITY_NAMES, SCORE_NAMES
from src.ml_utils import split_data, get_param_grids, tune_and_evaluate
from src.evaluation import plot_confusion, print_report
from src import setup_plot_style

setup_plot_style()

## 1. Chargement des embeddings DistilBERT

In [ ]:
X = np.load('../../outputs/distilbert_embeddings.npy')
y_polarity = np.load('../../outputs/distilbert_labels_polarity.npy')
y_score = np.load('../../outputs/distilbert_labels_score.npy')

print(f'Embeddings : {X.shape}')
print(f'\nPolarité :')
for label, name in enumerate(POLARITY_NAMES):
    n = (y_polarity == label).sum()
    print(f'  {label} ({name}): {n:,} ({n/len(y_polarity)*100:.1f}%)')
print(f'\nScore :')
for star in range(1, 6):
    n = (y_score == star).sum()
    print(f'  {star}★: {n:,} ({n/len(y_score)*100:.1f}%)')

## 2. Split 80/10/10 & Standardisation

Split stratifié sur la polarité, puis standardisation (importante pour LogReg et SVM).

In [ ]:
data = split_data(X, y_polarity, y_score)
print(f"Train: {len(data['X_train'])} | Val: {len(data['X_val'])} | Test: {len(data['X_test'])}")

# Standardisation (RF est scale-invariant, donc pas de souci)
scaler = StandardScaler()
X_train = scaler.fit_transform(data['X_train'])
X_val = scaler.transform(data['X_val'])
X_test = scaler.transform(data['X_test'])

print(f'Standardisation appliquée (mean={X_train.mean():.6f}, std={X_train.std():.6f})')

## 3. Classification — Polarité (3 classes)

In [ ]:
# GaussianNB car les embeddings sont continus (pas de MultinomialNB)
grids = get_param_grids(use_gaussian_nb=True)
results_pol, models_pol, preds_pol = tune_and_evaluate(
    grids, X_train, data['y_pol_train'], X_val, data['y_pol_val']
)

In [ ]:
results_pol_df = pd.DataFrame(results_pol).T
display(results_pol_df.sort_values('f1', ascending=False))

best_pol = results_pol_df['f1'].idxmax()
print(f"\nMeilleur modèle Polarité : {best_pol} (F1 Macro={results_pol_df.loc[best_pol, 'f1']:.4f})")

plot_confusion(data['y_pol_val'], preds_pol[best_pol], POLARITY_NAMES, f'{best_pol} — Polarité (LLM)')
print_report(data['y_pol_val'], preds_pol[best_pol], POLARITY_NAMES, f'{best_pol} — Polarité')

## 4. Classification — Score (1-5 étoiles)

In [ ]:
grids_sc = get_param_grids(use_gaussian_nb=True)
results_sc, models_sc, preds_sc = tune_and_evaluate(
    grids_sc, X_train, data['y_sc_train'], X_val, data['y_sc_val']
)

In [ ]:
results_sc_df = pd.DataFrame(results_sc).T
display(results_sc_df.sort_values('f1', ascending=False))

best_sc = results_sc_df['f1'].idxmax()
print(f"\nMeilleur modèle Score : {best_sc} (F1 Macro={results_sc_df.loc[best_sc, 'f1']:.4f})")

plot_confusion(data['y_sc_val'], preds_sc[best_sc], SCORE_NAMES, f'{best_sc} — Score (LLM)')
print_report(data['y_sc_val'], preds_sc[best_sc], SCORE_NAMES, f'{best_sc} — Score')

## 5. Test Final

In [ ]:
print_report(data['y_pol_test'], models_pol[best_pol].predict(X_test),
             POLARITY_NAMES, f'{best_pol} — Polarité (Test)')
print_report(data['y_sc_test'], models_sc[best_sc].predict(X_test),
             SCORE_NAMES, f'{best_sc} — Score (Test)')

print('\n✅ Notebook SAE-117 terminé')
print(f'   Meilleur Polarité : {best_pol} (F1 Macro={results_pol_df.loc[best_pol, "f1"]:.4f})')
print(f'   Meilleur Score    : {best_sc} (F1 Macro={results_sc_df.loc[best_sc, "f1"]:.4f})')